**04 · Mapas de calor, dispersión y robustez**

Ejecutar desde el repositorio point-field con `python -m pip install -e ".[learning,bench,dev]"` y Jupyter. Las salidas nuevas se guardan en `runs/tesis_figuras/`. No reemplazar resultados históricos sin comprobar el protocolo.

In [ ]:
from pathlib import Path
import sys, subprocess, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'benchmarks'/'common.py').exists() and (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz de point-field o una subcarpeta del repositorio.")
sys.path.insert(0, str(ROOT/'benchmarks'))
OUT = ROOT/'runs'/'tesis_figuras'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.dpi': 120, 'savefig.dpi': 300, 'pdf.fonttype': 42})
COLORS = ['#21618c','#c65036','#63933e','#8b65a5','#c49a38','#458e91']
def save(fig, name):
    for ext in ('pdf','png'):
        fig.savefig(OUT/f'{name}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
def points(ax, X, y=None):
    if y is None: ax.scatter(*X.T, s=5, c=COLORS[0], linewidths=0)
    else:
        for j,c in enumerate(np.unique(y)):
            ax.scatter(*X[y==c].T, s=5, c='#b9b9b9' if c==-1 else COLORS[j%len(COLORS)], linewidths=0)
    ax.set_xticks([]); ax.set_yticks([])
    if X.shape[1]==2: ax.set_aspect('equal', adjustable='box')
    else: ax.set_zticks([]); ax.set_box_aspect((1,1,1)); ax.view_init(elev=23, azim=35)
def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT/'benchmarks'/name), *map(str,args)], cwd=ROOT, check=True)
print('Repositorio:', ROOT, '\nSalidas:', OUT)
try:
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
except (OSError, subprocess.CalledProcessError):
    commit='No disponible'
print('Commit:',commit)

**1. Seleccionar una única fuente de resultados.** No mezclar ejecuciones nuevas e históricas. ARI puede ser negativo, por lo que la escala debe conservar esos valores.

In [ ]:
from common import MAIN_CASES,HELD_OUT_CASES,METHODS,VARIANTS
USE_NEW=False
source=OUT/'03_clustering_eval_nuevo.csv' if USE_NEW else ROOT/'benchmarks/results/clustering_2d_eval.csv'
df=pd.read_csv(source); assert {'case','variant','seed','method','ARI','NMI'}<=set(df.columns)
print(source); display(df.groupby(['case','variant','method']).seed.nunique().describe())
methods=[m for m in METHODS if m in df.method.unique()]
short=['K-Means','DBSCAN','OPTICS','Espectral','V0','V1','V2']; label=dict(zip(METHODS,short))
summary=df.groupby(['case','variant','method']).agg(ARI_mean=('ARI','mean'),ARI_sd=('ARI','std'),NMI_mean=('NMI','mean'),n=('seed','nunique')).reset_index()
summary.to_csv(OUT/'04_resumen.csv',index=False)

**2. Heatmaps separados.** Los casos principales y reservados se muestran por separado para mejorar legibilidad. Cada celda contiene media; la dispersión se guarda en CSV.

In [ ]:
from common import load_case
from matplotlib.colors import Normalize
for cases,tag in [(MAIN_CASES,'principales'),(HELD_OUT_CASES,'reservados')]:
    idx=[(c,v) for c in cases for v in VARIANTS]
    vals=summary.pivot(index=['case','variant'],columns='method',values='ARI_mean').reindex(index=pd.MultiIndex.from_tuples(idx),columns=methods).to_numpy()
    fig=plt.figure(figsize=(11,max(4, .47*len(idx)+1.3)),constrained_layout=True)
    gs=fig.add_gridspec(len(idx),2,width_ratios=[1,9]); ax=fig.add_subplot(gs[:,1]); norm=Normalize(vmin=min(0,np.nanmin(vals)),vmax=1);im=ax.imshow(vals,aspect='auto',cmap='RdYlGn',norm=norm)
    for i,(c,v) in enumerate(idx):
        mini=fig.add_subplot(gs[i,0]); ds=load_case(c,v,100);points(mini,ds.X,ds.y);mini.set_axis_off()
        for j,val in enumerate(vals[i]):
            if np.isfinite(val):ax.text(j,i,f'{val:.2f}',ha='center',va='center',color='white' if val<.18 or val>.82 else 'black',fontsize=9)
    ax.set_xticks(range(len(methods)),[label[m] for m in methods]);ax.xaxis.tick_top();ax.set_yticks(range(len(idx)),[f'{c} · {v}' for c,v in idx],fontsize=8)
    fig.colorbar(im,ax=ax,label='ARI medio');save(fig,'04_heatmap_'+tag)

**3. Pérdida de ARI por perturbación.** Diferencias pareadas dentro de la misma escena, método y semilla. La pérdida positiva representa deterioro.

In [ ]:
base=df[df.variant=='base'][['case','seed','method','ARI']].rename(columns={'ARI':'ARI_base'})
delta=df[df.variant!='base'].merge(base,on=['case','seed','method'],validate='many_to_one');delta['perdida']=delta.ARI_base-delta.ARI
a=delta.groupby(['method','variant']).perdida.agg(['mean','std']).reset_index();a.to_csv(OUT/'04_perdida_ari.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4),constrained_layout=True)
for j,v in enumerate(VARIANTS[1:]):
    d=a[a.variant==v].set_index('method').reindex(methods);ax.bar(np.arange(len(methods))+(j-1)*.23,d['mean'],width=.22,label=v,yerr=d['std'],capsize=2)
ax.axhline(0,color='black',lw=.7);ax.set_xticks(range(len(methods)),[label[m] for m in methods]);ax.set_ylabel('ARI base − ARI perturbado');ax.legend(frameon=False);save(fig,'04_robustez')